# Estrutura Completa do Projeto — Análise de Cashback
### Visão Geral da Metodologia
O projeto segue o framework CRISP-DM adaptado para análise de retenção:

Entendimento do Negócio → Preparação dos Dados → EDA → Modelagem → Avaliação → Comunicação

## Decomposição do projeto
### **Etapa 1 — Coleta e Entendimento dos Dados**

Dataset principal: Online Retail (UCI / Kaggle)
- ~540k transações, campos: InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country

O que mapear antes de tocar nos dados:

- Granularidade: uma linha = um item de uma transação
- Identificador de cliente: CustomerID (equivalente ao CPF simulado)
- Janela temporal disponível: ~1 ano (dez/2010 a dez/2011)
- Ausências críticas: ~25% dos registros sem CustomerID — decisão explícita de exclusão

---

### **Etapa 2 — Limpeza (Python / Pandas)**
Checklist de limpeza
- Remover registros sem CustomerID
- Filtrar Quantity <= 0 (devoluções — tratar separado ou excluir)
- Filtrar UnitPrice <= 0
- Converter InvoiceDate para datetime
- Criar coluna TotalPrice = Quantity * UnitPrice
- Remover duplicatas exatas
- Filtrar Country == 'United Kingdom' (foco e consistência)

---

### **Etapa 3 — Feature Engineering (SQL + Python)**

**Camada 1 — Agregação e Estrutura Temporal (DuckDB / SQL)**

Elevar as transações do nível de item para o nível de pedido e
enriquecer cada registro com o contexto necessário para aplicar as regras
na camada seguinte.

- Agregar as transações para o nível de pedido
- Via window functions, puxar para cada pedido as informações do pedido
  imediatamente anterior do mesmo cliente
- Calcular o bônus bruto gerado pela compra anterior
- Organizar todas as transformações em CTEs encadeadas para separar
  claramente a intenção de cada etapa

**Camada 2 — Aplicação das Regras de Cashback (Python / Pandas)**

Percorrer os pedidos de cada cliente,
mantendo um saldo de bônus atualizado a cada iteração e classificando
cada pedido conforme as regras de negócio.


- Classificação de cada pedido: `primeira_compra`, `mesmo_dia`, 
`expirado`, `resgatado_total` e `resgatado_parcial`

- Cálculos por pedido: `bonus_disponivel`, `bonus_aplicado`, 
`bonus_perdido` e `bonus_gerado`

- Simulações de cenário de cashback para janelas de 30, 45 e 60 dias

---

### **Etapa 4 — EDA (Python / Pandas + Plotly)**

**Segmentação RFM (base para todas as análises seguintes)**

- Recência, Frequência e Valor monetário por cliente
- Classificar em 3 faixas: Alto, Médio, Baixo
- Atribuir rótulo de segmento (Champions, Em risco, Inativos)
- Incorporar o segmento como coluna na tabela analítica — a partir daqui ele é só uma variável de corte

**Intervalo entre compras por segmento**

- Distribuição de days_since_last por segmento RFM — onde está a massa em relação à janela de 30 dias?
- Proporção de pedidos com status expirado vs. resgatados por segmento — a expiração é generalizada ou concentrada?

**Impacto financeiro da expiração por segmento**

- Razão entre bonus_perdido e bonus_gerado por segmento — quem está desperdiçando mais cashback?
- Identificar se a perda está concentrada em clientes de médio valor, o que eleva o risco de churn

**Comparação entre cenários (30 / 45 / 60 dias) por segmento**

- Como a taxa de expiração muda entre os três cenários para cada segmento?
- Qual janela efetivamente aumenta o bonus_aplicado — e para quais perfis isso é mais relevante?

---

### **Etapa 5 — Análise de Coortes**
Coorte = mês da primeira compra do cliente

**O que analisar:**
- Taxa de retenção por coorte ao longo dos meses
- Comparar retenção hipotética com cashback 30d vs. 45d vs. 60d (usando os flags criados)
- Identificar o "cliff" — onde a retenção cai mais abruptamente

---

### **Etapa 6 — Simulação de Cenários**

**Tabela de trade-off agregada**
- Para cada janela de validade (30, 45, 60 dias), calcular por segmento RFM:
  - Taxa de expiração (bonus_perdido / bonus_gerado)
  - Taxa de resgate (bonus_aplicado / bonus_gerado)
  - Custo total do programa (bonus_aplicado acumulado)
- Organizar em uma tabela comparativa segmento × cenário — esse é o entregável central da etapa

**Cálculo do impacto incremental**
- Quanto cada extensão de prazo (30→45, 45→60) reduz a expiração por segmento?
- Quanto aumenta o custo do programa em cada salto?
- Identificar o ponto de retorno marginal decrescente — onde ampliar a janela deixa de gerar ganho relevante

**Visualização de trade-off (Retenção × Custo)**
- Gráfico de linhas ou barras agrupadas por cenário e segmento, mostrando simultaneamente taxa de resgate e custo
- Destaque visual para o segmento Em risco — é onde a mudança de janela tem maior potencial de impacto de retenção
---

### **Etapa 7 — Dashboard (Power BI)**

**3 páginas com progressão narrativa: problema → causa → decisão**

| Página | Pergunta central | Visuais principais |
|---|---|---|
| Visão Geral | O programa tem um problema de expiração relevante? | KPIs de bônus gerado / aplicado / expirado; taxa de expiração por janela e segmento; expiração vs. resgate por segmento |
| Diagnóstico | Onde está o problema e por que acontece? | Taxa de expiração por janela e segmento; bônus gerado vs. resgatado por janela; tabela de frequência média e taxas por segmento |
| Recomendações | O que fazer — e para quem? | Clientes recuperados por extensão de janela; custo incremental por extensão; eficiência marginal por segmento; cards de recomendação por segmento |

---

### **Etapa 8 — Conclusões e Recomendações**

- Qual janela representa o melhor equilíbrio entre custo do programa e resgate para cada segmento?
- Existe um cenário em que uma janela diferenciada por perfil RFM faz sentido — ou uma janela única resolve bem?
- Síntese em uma ou duas frases de negócio, direto aplicável como recomendação para stakeholders


